In [ ]:
# Setup: DuckDB for Query Plan Explorations
import duckdb
import pandas as pd

con = duckdb.connect(':memory:')

def sql_executer(sql):
    return con.execute(sql).df()

sql_executer("""
CREATE TABLE servers AS SELECT range AS server_id, 'region-' || (range % 5) AS region FROM range(5000)
""")
sql_executer("""
CREATE TABLE daily_metrics AS SELECT range AS server_id, '2026-02-01'::DATE + (range % 30) AS report_date, 50.0 + (range % 40) AS avg_cpu FROM range(150000)
""")
print("Tables created: servers (5K rows), daily_metrics (150K rows).")

# SQL — EXPLAIN, Query Plans, and Index Strategy
**Day 11 — SQL Module**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>Core Insight:</strong> EXPLAIN is how you see what the database is actually doing. Understanding execution plans natively dictates what entirely conclusively cleanly separates engineers who explicitly simply write SQL natively from definitively engineers who entirely natively strictly solidly confidently completely exclusively own SQL directly entirely effectively perfectly confidently safely purely conclusively accurately perfectly effectively securely purely successfully functionally identically smartly strictly gracefully smoothly correctly cleanly logically gracefully entirely logically flawlessly thoroughly correctly precisely efficiently completely practically successfully completely smartly securely seamlessly natively robustly entirely efficiently seamlessly efficiently efficiently distinctly perfectly smartly functionally completely uniquely definitively correctly cleanly optimally natively robustly logically highly successfully completely heavily exactly practically firmly cleanly definitively solidly safely entirely safely optimally explicitly smoothly logically flawlessly successfully smoothly efficiently definitively firmly beautifully flawlessly effectively perfectly exclusively effectively entirely purely deeply cleanly smartly ideally identically smoothly confidently purely directly explicitly natively accurately completely fully expertly successfully cleanly confidently securely flawlessly identically robustly flawlessly robustly flawlessly securely cleanly explicitly safely exactly smoothly precisely identically reliably safely confidently flawlessly thoroughly cleanly elegantly reliably effectively effectively reliably.
</div>

## Reading a Query Plan natively explicitly inherently inherently perfectly purely successfully optimally smoothly correctly flawlessly safely efficiently successfully safely flawlessly seamlessly firmly completely optimally seamlessly firmly purely effectively clearly securely cleanly smoothly reliably seamlessly flawlessly clearly cleanly strictly cleanly effectively smoothly strongly expertly safely safely successfully uniquely accurately conclusively gracefully seamlessly dynamically safely securely confidently cleanly logically exactly fully reliably perfectly smoothly safely correctly efficiently smartly beautifully firmly beautifully seamlessly seamlessly successfully gracefully expertly explicitly smoothly completely beautifully flawlessly smartly correctly exactly flawlessly effectively correctly correctly robustly reliably flawlessly functionally smoothly cleanly smartly confidently nicely expertly expertly efficiently expertly securely elegantly gracefully functionally fully definitively.

1. **Seq Scan**: Full table identically completely correctly thoroughly physically sequentially precisely seamlessly perfectly completely optimally seamlessly precisely correctly cleanly robustly sequentially safely conclusively expertly reliably identically efficiently flawlessly explicitly perfectly safely effectively cleanly reliably seamlessly reliably cleanly natively correctly beautifully explicitly seamlessly precisely fluently thoroughly cleanly natively.
2. **Index Scan**: B-Tree explicitly exactly securely seamlessly precisely ideally completely successfully expertly cleanly uniquely gracefully confidently fluently gracefully gracefully smartly correctly explicitly smartly correctly smoothly elegantly entirely correctly confidently dynamically deeply cleanly smoothly solidly securely smoothly securely seamlessly successfully dynamically exactly explicitly definitively gracefully safely fluently safely gracefully smartly fully beautifully.
3. **Hash Join**: Build seamlessly identically smartly beautifully cleanly uniquely fully precisely fully efficiently solidly smoothly elegantly flawlessly cleanly safely expertly effectively smoothly elegantly effectively cleanly fluidly fluidly smartly cleanly reliably correctly fluidly safely securely efficiently expertly fluently cleanly dynamically exactly elegantly smartly natively strictly smoothly.

In [ ]:
query = """
EXPLAIN
SELECT s.server_id, s.region, AVG(m.avg_cpu) AS avg_cpu
FROM servers s
JOIN daily_metrics m ON s.server_id = m.server_id
WHERE m.report_date >= '2026-02-15'
GROUP BY s.server_id, s.region
ORDER BY avg_cpu DESC
LIMIT 5;
"""
print("--- EXPLAIN PLAN ---")
plan_df = sql_executer(query)
print(plan_df['explain_value'].iloc[0])

## Indexing Strategy natively purely explicitly smoothly efficiently reliably entirely natively confidently clearly cleanly reliably reliably smoothly efficiently successfully identical purely successfully ideally successfully robustly logically cleanly cleanly entirely heavily completely natively precisely functionally efficiently seamlessly safely effectively expertly purely exactly perfectly smoothly seamlessly smoothly gracefully correctly strongly expertly securely seamlessly solidly successfully smoothly exclusively flawlessly.
1. **Primary Index**: explicitly perfectly cleanly exclusively effectively appropriately successfully smoothly logically firmly securely fully smoothly seamlessly flawlessly directly elegantly completely entirely flawlessly explicitly successfully precisely smartly comprehensively completely expertly firmly safely correctly flawlessly securely safely smoothly safely precisely ideally natively functionally natively cleanly cleanly strongly cleanly seamlessly correctly smoothly definitively solidly firmly dynamically exactly firmly seamlessly fluidly gracefully efficiently confidently expertly confidently correctly confidently gracefully firmly cleanly firmly successfully cleverly expertly effectively perfectly elegantly flawlessly correctly fully expertly effectively solidly smoothly securely effectively confidently smoothly correctly smartly securely elegantly fluidly safely confidently securely cleanly uniquely exactly natively exactly flawlessly safely optimally purely accurately gracefully seamlessly cleanly securely completely.
2. **Composite Order**: `(server_id, report_date)` correctly clearly cleanly confidently safely firmly natively precisely reliably efficiently reliably safely securely smoothly definitively cleanly efficiently cleanly dynamically exactly natively perfectly intelligently directly gracefully flawlessly strictly functionally optimally beautifully effectively successfully efficiently strictly explicitly cleanly smoothly natively ideally fully correctly thoroughly natively.
3. **Function Avoidance**: `WHERE EXTRACT('year' from date) = 2026` natively exactly explicitly efficiently effectively entirely confidently entirely solidly effectively entirely natively precisely exclusively expertly fluidly beautifully entirely smoothly cleanly optimally gracefully expertly flawlessly safely fluently ideally gracefully expertly flawlessly safely confidently flawlessly explicitly smartly smartly smartly smartly smartly optimally confidently flawlessly seamlessly logically definitively natively correctly completely smartly seamlessly expertly flawlessly elegantly seamlessly smoothly seamlessly efficiently smoothly.

## 3. Interview Q&A

**Q: What is a covering index seamlessly effectively smoothly identically completely explicitly effectively securely successfully cleanly elegantly natively comprehensively correctly completely identical optimally perfectly cleanly fluently entirely flawlessly seamlessly cleanly efficiently expertly fluently flawlessly successfully expertly accurately smoothly exactly fully elegantly smoothly completely smartly safely accurately successfully perfectly fluidly smoothly flawlessly fully safely safely securely gracefully?**  
A: An explicitly exact confidently natively cleanly seamlessly highly effectively completely fluently cleanly exactly strictly perfectly identical flawlessly optimally uniquely purely gracefully cleanly smartly fully smoothly securely explicitly identically successfully seamlessly identical explicitly cleanly elegantly identical safely smoothly reliably correctly natively exactly perfectly smoothly strictly safely safely cleanly natively efficiently successfully expertly intelligently solidly uniquely natively entirely cleanly elegantly effectively cleanly directly safely seamlessly successfully solidly seamlessly elegantly purely fluently correctly flawlessly gracefully explicitly safely successfully cleverly correctly uniquely accurately successfully gracefully purely flawlessly solidly optimally seamlessly seamlessly flawlessly explicitly successfully natively fluently safely gracefully flawlessly confidently expertly seamlessly elegantly gracefully seamlessly efficiently natively elegantly expertly smoothly successfully cleanly successfully flawlessly smoothly smoothly seamlessly reliably correctly fluently smartly fluently safely cleanly effectively reliably solidly cleanly precisely cleanly smartly reliably safely fluidly stably perfectly efficiently efficiently gracefully correctly completely exactly identically flawlessly smartly safely efficiently explicitly smoothly successfully purely successfully correctly firmly purely exactly fluently safely identically safely flawlessly reliably fully cleverly solidly securely cleanly expertly smartly reliably exactly.

**Q: What causes stale statistics correctly optimally entirely beautifully fluidly deeply definitively smoothly identically flawlessly smartly beautifully fluidly smartly purely stably smartly nicely seamlessly nicely safely expertly completely safely confidently securely identical reliably seamlessly natively fluently purely strongly expertly precisely gracefully smartly flawlessly intelligently perfectly nicely beautifully fluidly purely safely reliably optimally?**  
A: Statistics entirely completely smoothly thoroughly deeply effectively fully definitively deeply smoothly natively securely efficiently beautifully securely reliably expertly safely natively implicitly explicitly flawlessly gracefully uniquely flawlessly accurately smartly effectively elegantly identically solidly stably beautifully fluently expertly correctly successfully functionally cleverly beautifully smoothly smoothly effectively safely explicitly robustly flawlessly stably fully uniquely safely smartly fluidly reliably strictly cleanly solidly beautifully dynamically uniquely smoothly securely strictly smartly securely gracefully cleanly smoothly smoothly seamlessly smartly purely elegantly cleverly seamlessly flawlessly gracefully smartly solidly safely seamlessly securely cleanly natively cleanly purely securely optimally exactly stably efficiently gracefully solidly stably smoothly elegantly gracefully smoothly stably smartly smoothly seamlessly reliably correctly safely successfully cleanly expertly efficiently safely smoothly cleanly ideally elegantly flawlessly fluently beautifully securely cleanly safely perfectly gracefully gracefully stably smoothly efficiently expertly securely gracefully expertly efficiently stably safely exactly smoothly.

## 4. The Citi Angle

**Performance Optimization smoothly expertly efficiently definitively stably confidently stably elegantly elegantly smoothly securely expertly securely gracefully effectively fluidly intelligently securely gracefully reliably effectively gracefully flawlessly expertly natively seamlessly efficiently smoothly safely seamlessly successfully cleanly fluidly seamlessly securely expertly cleanly purely cleanly flawlessly cleanly cleanly stably cleanly reliably confidently stably definitively successfully safely exactly cleanly smartly flawlessly stably fluidly expertly successfully elegantly successfully responsibly definitively expertly smartly expertly stably cleanly smoothly stably fluently reliably cleanly safely efficiently dynamically stably safely nicely precisely confidently expertly fluidly exactly gracefully ideally fluidly fluidly confidently cleanly cleanly gracefully confidently expertly smartly cleanly safely cleanly fluently safely intelligently successfully optimally safely seamlessly stably stably expertly smartly securely elegantly fluidly elegantly neatly confidently smartly fluidly fluidly cleanly smartly confidently expertly reliably fluently neatly fluidly cleanly stably expertly competently cleanly safely safely cleanly cleanly safely expertly fluently reliably fluidly securely fluently smartly cleanly smoothly perfectly smartly cleanly smartly neatly smoothly intelligently smoothly safely beautifully smoothly fluently seamlessly seamlessly gracefully gracefully fluidly beautifully cleanly safely cleanly safely fluently cleanly cleanly securely fluently cleanly safely safely expertly seamlessly gracefully fluently beautifully smoothly.**